# CNN on CSV Upload Data

## Objective
Train CNN from uploaded flattened-image CSV (label in first column, pixels in remaining columns).

## Google Colab Steps
1. Run the setup cell.
2. Execute the main cell.
3. Save a copy to your Google Drive if needed.
## Notes
Expected CSV format: first column = class label, remaining columns = flattened grayscale pixels.



In [ ]:
!pip -q install numpy pandas matplotlib seaborn scikit-learn tensorflow


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from google.colab import files
from sklearn.model_selection import train_test_split

uploaded = files.upload()
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)
display(df.head())

label_col = df.columns[0]
y = df[label_col].astype(int).to_numpy()
X = df.drop(columns=[label_col]).to_numpy(dtype=np.float32)

side = int(np.sqrt(X.shape[1]))
if side * side != X.shape[1]:
    raise ValueError("Pixel count must form a square image (e.g., 784 for 28x28).")

X = (X / 255.0).reshape(-1, side, side, 1)
num_classes = len(np.unique(y))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = models.Sequential([
    layers.Conv2D(16, (3, 3), activation="relu", input_shape=(side, side, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(num_classes, activation="softmax"),
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test), verbose=1)

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print("Test accuracy:", acc)
